# Applio — экспериментальные вокодеры

Этот блокнот запускает fork `egor125552/Applio`, ветку `exp/vocoders`. Запускай ячейки сверху вниз.


In [ ]:
# @title 1. Установить Applio Vocoders
rebuild_environment = False  # @param {type:"boolean"}

from pathlib import Path
import shutil, subprocess, sys

REPO_URL = "https://github.com/egor125552/Applio.git"
BRANCH = "exp/vocoders"
APP_DIR = Path("/content/Applio-vocoders")
VENV_DIR = Path("/content/applio-vocoders-env")
PYTHON = VENV_DIR / "bin" / "python"
MARKER = VENV_DIR / ".applio_commit"
REQ_FILE = Path("/content/applio-vocoders-requirements.txt")

if not Path("/content").exists():
    raise RuntimeError("Этот блокнот рассчитан на Google Colab.")

def run(command, cwd=None):
    command = [str(x) for x in command]
    print("\n$", " ".join(command), flush=True)
    subprocess.run(command, cwd=cwd, check=True)

has_gpu = shutil.which("nvidia-smi") is not None
if has_gpu:
    run(["nvidia-smi"])
else:
    print("⚠️ GPU не найден: приложение запустится, но будет медленным.")

run(["apt-get", "update", "-qq"])
run(["apt-get", "install", "-y", "-qq", "git", "git-lfs", "ffmpeg", "libsndfile1", "portaudio19-dev", "build-essential"])

if (APP_DIR / ".git").exists():
    run(["git", "remote", "set-url", "origin", REPO_URL], APP_DIR)
    run(["git", "fetch", "--depth", "1", "origin", BRANCH], APP_DIR)
    run(["git", "checkout", "-B", BRANCH, "FETCH_HEAD"], APP_DIR)
else:
    if APP_DIR.exists():
        shutil.rmtree(APP_DIR)
    run(["git", "clone", "--depth", "1", "--branch", BRANCH, "--single-branch", REPO_URL, APP_DIR])

run(["git", "lfs", "pull"], APP_DIR)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=APP_DIR, text=True).strip()
print("Commit:", commit)

run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "uv"])
UV = [sys.executable, "-m", "uv"]
current = PYTHON.exists() and MARKER.exists() and MARKER.read_text().strip() == commit

if rebuild_environment or not current:
    if VENV_DIR.exists():
        shutil.rmtree(VENV_DIR)
    run(UV + ["python", "install", "3.10"])
    run(UV + ["venv", "--seed", "--python", "3.10", VENV_DIR])
    run([PYTHON, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"])

    torch_index = "https://download.pytorch.org/whl/cu121" if has_gpu else "https://download.pytorch.org/whl/cpu"
    run([PYTHON, "-m", "pip", "install", "--no-cache-dir", "--index-url", torch_index, "torch==2.3.1", "torchvision==0.18.1", "torchaudio==2.3.1"])

    skipped = {"torch==2.3.1", "torchvision==0.18.1", "torchaudio==2.3.1", "treon"}
    requirements = (APP_DIR / "requirements.txt").read_text(encoding="utf-8").splitlines()
    requirements = [line for line in requirements if line.strip().lower() not in skipped]
    requirements += ["", "# Colab compatibility", "pydantic==2.8.2", "fastapi==0.112.0", "starlette==0.37.2"]
    REQ_FILE.write_text("\n".join(requirements) + "\n", encoding="utf-8")
    run([PYTHON, "-m", "pip", "install", "--no-cache-dir", "-r", REQ_FILE])
    run([PYTHON, "-m", "pip", "check"])
    MARKER.write_text(commit + "\n")
else:
    print("✅ Окружение уже установлено для этого commit.")

print("✅ Установка завершена.")


In [ ]:
# @title 2. Проверить код и импорты вокодеров
from pathlib import Path
import subprocess, textwrap
APP_DIR = Path("/content/Applio-vocoders")
PYTHON = Path("/content/applio-vocoders-env/bin/python")
if not PYTHON.exists():
    raise RuntimeError("Сначала запусти установочную ячейку.")
test = r'''
import compileall, importlib, torch
if not compileall.compile_dir('.', quiet=1):
    raise RuntimeError('Обнаружена синтаксическая ошибка')
modules = [
 'rvc.lib.algorithm.generators.bigvgan',
 'rvc.lib.algorithm.generators.ddsp',
 'rvc.lib.algorithm.generators.ddsp_v2',
 'rvc.lib.algorithm.generators.ddsp_v3',
 'rvc.lib.algorithm.generators.hifigan',
 'rvc.lib.algorithm.generators.hifigan_aa',
 'rvc.lib.algorithm.generators.hifigan_cam',
 'rvc.lib.algorithm.generators.hifigan_nsf',
 'rvc.lib.algorithm.generators.hifigan_pqmf',
 'rvc.lib.algorithm.generators.hifigan_snake',
 'rvc.lib.algorithm.generators.hiftnet',
 'rvc.lib.algorithm.generators.hiftnet2',
 'rvc.lib.algorithm.generators.ringformer',
 'rvc.lib.algorithm.generators.velocity',
 'rvc.lib.algorithm.generators.vocos',
 'rvc.lib.algorithm.generators.vocos_v2',
 'rvc.lib.algorithm.generators.wavehax',
 'rvc.lib.algorithm.generators.wavehax1d',
 'rvc.lib.algorithm.generators.wavenext',
 'rvc.lib.algorithm.discriminators.CoMBD',
 'rvc.lib.algorithm.discriminators.SBD',
 'rvc.lib.algorithm.discriminators.mbd',
 'rvc.lib.algorithm.discriminators.mpd',
 'rvc.lib.algorithm.discriminators.mrd',
 'rvc.lib.algorithm.discriminators.msd',
 'rvc.lib.algorithm.discriminators.univnet',
]
errors = []
for name in modules:
    try:
        importlib.import_module(name); print('OK ', name)
    except Exception as error:
        errors.append((name, repr(error))); print('ERR', name, repr(error))
print('torch:', torch.__version__, 'cuda:', torch.version.cuda)
if errors:
    raise RuntimeError('Не прошли импорты: ' + repr(errors))
'''
subprocess.run([str(PYTHON), "-c", textwrap.dedent(test)], cwd=APP_DIR, check=True)
print("✅ Проверка вокодеров пройдена.")


In [ ]:
# @title 3. Синхронизировать модель с Google Drive (необязательно)
sync_mode = "не синхронизировать"  # @param ["не синхронизировать", "Drive → Colab", "Colab → Drive"]
model_name = ""  # @param {type:"string"}
from pathlib import Path
import shutil
if sync_mode != "не синхронизировать":
    from google.colab import drive
    drive.mount("/content/drive")
    if not model_name.strip():
        raise ValueError("Укажи имя модели.")
    local = Path("/content/Applio-vocoders/logs") / model_name.strip()
    remote = Path("/content/drive/MyDrive/ApplioBackup") / model_name.strip()
    source, destination = (remote, local) if sync_mode == "Drive → Colab" else (local, remote)
    if not source.exists():
        raise FileNotFoundError(source)
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source, destination, dirs_exist_ok=True)
    print("✅ Синхронизация завершена:", destination)
else:
    print("Синхронизация пропущена.")


In [ ]:
# @title 4. Запустить Applio
share_link = True  # @param {type:"boolean"}
port = 6969  # @param {type:"integer"}
from pathlib import Path
import os, subprocess
APP_DIR = Path("/content/Applio-vocoders")
PYTHON = Path("/content/applio-vocoders-env/bin/python")
if not (APP_DIR / "app.py").exists() or not PYTHON.exists():
    raise RuntimeError("Сначала запусти установочную ячейку.")
command = [str(PYTHON), "app.py", "--port", str(port)]
if share_link:
    command.append("--share")
environment = os.environ.copy()
environment["PYTHONUNBUFFERED"] = "1"
environment["TF_CPP_MIN_LOG_LEVEL"] = "3"
print("$", " ".join(command))
print("Ячейка работает, пока запущен сервер. Stop остановит сервер.\n")
subprocess.run(command, cwd=APP_DIR, env=environment, check=True)
